# 03 - Browse the therapeutic-hypothesis annex

The hypothesis layer is **hypothesis-generating**: prioritized, tiered
leads with explicit null models and stated kill/confirm experiments.
Confidence tiers: **A** anchor-validated (multiple independent evidence
lines converge), **B** strong pooled signal with nulls addressed,
**C** triage. This notebook tours the tiers, the flagship HSF1 lead, and
the kill/confirm column. Read `annex_hypotheses/README.md` and
`HOW_TO_READ.md` alongside any table.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the package root (works whether the notebook runs from examples/
# or from the package root).
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "core" / "basis" / "basis_registry.json").exists())
sys.path.insert(0, str(ROOT / "src"))
print("package root located (all paths below are relative to it)")


package root located (all paths below are relative to it)


## The tables and the tiers

In [2]:
ANNEX = ROOT / "annex_hypotheses"
ledger = pd.read_csv(ANNEX / "hypothesis_ledger_full.csv")
leads = pd.read_csv(ANNEX / "anchor_leads.csv")
atlas = pd.read_csv(ANNEX / "program_atlas.csv")
sharp = pd.read_csv(ANNEX / "sharp_sar_candidates.csv")

print(f"hypothesis_ledger_full: {len(ledger)} rows (triage-grade, hypothesis-generating)")
print(ledger["status"].value_counts().to_string())
print()
print("ledger confidence tiers:", ledger["confidence_tier"].value_counts().to_dict())
print(f"anchor_leads: {len(leads)} rows | program_atlas: {len(atlas)} rows "
      f"| sharp_sar_candidates: {len(sharp)} rows")


hypothesis_ledger_full: 1027 rows (triage-grade, hypothesis-generating)
status
hypothesis_triage              978
hypothesis_strong               47
hypothesis_anchor_validated      2

ledger confidence tiers: {'C': 978, 'B': 47, 'A': 2}
anchor_leads: 16 rows | program_atlas: 26 rows | sharp_sar_candidates: 133 rows


## The flagship: the HSF1 heat-shock chemotype (ZSH-0001)

Tier A. In `zel039_aec7`, compounds carrying bb0 level `BB_2085420374`
induce a canonical HSF1/heat-shock program that reproduces the measured
phenotype of the control HTH-01-015 at rank 1. Three independent lines
converge; the caveat flag `weak_null` records that specificity lives in
the rank-1 match identity, not in the p-value (see
`annex_hypotheses/HOW_TO_READ.md` for the weak-null rule).

In [3]:
flagship = leads[leads["hypothesis_id"] == "ZSH-0001"].iloc[0]
for field in ["hypothesis_id", "bb_level_ids", "contexts", "biological_program",
              "matched_control", "moa_class", "confidence_tier", "caveats"]:
    print(f"{field:20s} {flagship[field]}")
print()
print("evidence_summary:")
print(flagship["evidence_summary"])


hypothesis_id        ZSH-0001
bb_level_ids         bb0=BB_2085420374
contexts             zel039_aec7 (primary; HSR program present in hek293/a549 controls at ~5x lower amplitude)
biological_program   HSF1 / heat-shock response (HSPA1A/B, HSPA6, DNAJB1, HSPD1, HSPH1, HSP90AA1, CRYAB)
matched_control      HTH-01-015
moa_class            HSF1_heat_shock_validated (panel annotation: NUAK1/ROCK class)
confidence_tier      A
caveats              weak_null

evidence_summary:
Control's pure-measurement signature is the panel's most extreme heat-shock response (mean z=+3.94 across 9 HSP genes; next control +1.83); additive bb-effect vector correlates with the control measurement at r=0.53 over 2,498 shared HVGs (rank 3 of ~260 level effect vectors); cos=0.631, p=0.000 vs 200-draw gene-label-permutation null (weak null: 88-98% of levels match SOME control at p<=0.01 under this gene-label permutation; specificity lives in the rank-1 match identity, not the p-value); 49/50 top-50 neighbors carry 

## The kill/confirm column

Every anchor lead ships the concrete experiment that would kill or
confirm it. This is the intended next step for each distilled lead.

In [4]:
for row in leads.itertuples(index=False):
    print(f"[{row.confidence_tier}] {row.hypothesis_id} - {row.biological_program.split('(')[0].strip()}")
    print(f"    kill/confirm: {str(row.kill_confirm_experiment)[:180]}...")
    print()


[A] ZSH-0001 - HSF1 / heat-shock response
    kill/confirm: (i) Pull raw pseudobulks of BB_2085420374 carriers vs non-carriers matched on the other bb positions and confirm HSPA1A/DNAJB1 induction in the unsmoothed data. (ii) HSF1-reporter ...

[A] ZSH-0002 - CDK4/6-inhibition / cell-cycle + translation axis
    kill/confirm: (i) Pull raw pseudobulks of BB_5422857344 carriers vs non-carriers matched on the other bb positions and confirm the claimed program in the unsmoothed data; (ii) cell-cycle cytomet...

[B] ZSH-3755 - HSF1 / heat-shock grammar program
    kill/confirm: Same raw-pseudobulk check as ZSH-0001 (i): carriers vs matched non-carriers; the bb effect vectors make direct gene-level predictions (HSPA1A/HSPA6/DNAJB1 up)...

[B] ZSH-3756 - RNA-processing / splicing-lineage program
    kill/confirm: Measure 3-5 BB_9076436922-carrying compounds (they exist in zel028 measured and zel031) in hek293/a549 and test the MALAT1/HNRNPU/SF3B1 program in raw data; if measured carriers do..

## Filtering the full ledger

Filter by `status` / `confidence_tier` first; the `hypothesis_triage`
rows (95.2% of the table) are raw material for re-ranking with your own
priors. Recorded nulls are kept on purpose: a good null is as useful as
a positive. Some entries will be false positives even with correct
nulls; that is the price of a complete ledger, and it is why the
distilled tables exist.

In [5]:
tier_ab = ledger[ledger["confidence_tier"].isin(["A", "B"])]
print(f"tier A+B ledger rows: {len(tier_ab)}")
print(tier_ab[["hypothesis_id", "hypothesis_type", "context", "confidence_tier"]]
      .head(12).to_string(index=False))


tier A+B ledger rows: 49
hypothesis_id       hypothesis_type       context confidence_tier
     ZSH-0001 level_control_mimicry   zel039_aec7               A
     ZSH-0002 level_control_mimicry   zel039_aec7               A
     ZSH-0003       pair_hypothesis zel024_hek293               B
     ZSH-0004       pair_hypothesis   zel028_a549               B
     ZSH-0005       pair_hypothesis zel024_hek293               B
     ZSH-0006       pair_hypothesis   zel028_a549               B
     ZSH-0007       pair_hypothesis zel024_hek293               B
     ZSH-0008       pair_hypothesis zel024_hek293               B
     ZSH-0009       pair_hypothesis zel024_hek293               B
     ZSH-0010       pair_hypothesis zel024_hek293               B
     ZSH-0011       pair_hypothesis   zel039_aec7               B
     ZSH-0012       pair_hypothesis zel024_hek293               B
